In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import mode
import scanpy as sc

In [3]:
# -*- coding: utf-8 -*-
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

#def build_paths(base_path, ids, extension):
    #return [base_path + id + extension for id in ids]

#ids = "151507 151508 151509 151510 151669 151670 151671 151672 151673 151674 151675 151676".split()
#image_paths = build_paths("/sibcb1/chenluonanlab8/zoujiawei/Final version with foundation model/data/DLPFC/image/", ids, "_full_image.tif")

def build_paths(base_path, ids, extension, remove_prefix=None):
    """Builds file paths, optionally removing a prefix from IDs.

    Args:
        base_path: Base directory path (should end with '/').
        ids: List of IDs (strings).
        extension: File extension (including the leading dot '.').
        remove_prefix: Prefix to remove from IDs (string, or None if no removal).

    Returns:
        List of file paths.  Returns an empty list if input is invalid or an error occurs.
    """
    try:

        cleaned_ids = [id.replace(remove_prefix, "") if remove_prefix else id for id in ids]

        return [base_path + id + extension for id in cleaned_ids]

    except Exception as e:
        print(f"An error occurred: {e}")
        return []


In [4]:
#eg cccrcc
ids = "H1 H2 H3 F1 F2 F3 E1 E2 E3 D1 D2 D3 D4 D5 D6 C1 C2 C3 C4 C5 C6 B1 B2 B3 B4 B5 B6 A1 A2 A3".split()

reduced_mtx_paths = build_paths("/sibcb1/chenluonanlab8/zoujiawei/Final version with foundation model/data/BRCA/st/oricount_", ids, ".csv")
image_paths = build_paths("/sibcb1/chenluonanlab8/zoujiawei/Final version with foundation model/data/BRCA/image_rename/standardized_", ids, ".jpg")
spatial_pos_paths = build_paths("/sibcb1/chenluonanlab8/zoujiawei/Final version with foundation model/data/BRCA/st/spot_", ids, ".csv")
#reduced_mtx_paths = build_paths("/sibcb1/chenluonanlab8/zoujiawei/Final version with foundation model/data/BRCA/st/intdata/", ids, ".csv")
barcode_paths = build_paths("/sibcb1/chenluonanlab8/zoujiawei/Final version with foundation model/data/BRCA/st/barcode_", ids, ".csv")

In [5]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

# 假设您的CSV文件都在同一个目录下
#data_dir = '/sibcb1/chenluonanlab8/zoujiawei/data_lung/10x/10x/'
csv_files = reduced_mtx_paths

# 创建空列表存储adata对象
adata_list = []


In [6]:
# 读取并处理每个CSV文件
for i, file in enumerate(csv_files):
    file_path =  file
    # 读取CSV并转置
    adata = sc.read_csv(file_path).T
    
    # 添加batch信息
    adata.obs['batch'] = f'batch_BRCA_{i}'
    
    # 基础预处理
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    adata.var['Gene Symbol'] = adata.var.index.values
    
    adata_list.append(adata)

In [7]:
# 合并所有数据集
adata_combined = adata_list[0].concatenate(adata_list[1:], join='inner', batch_key='batch')


In [8]:
adata_combined.X.shape

(11214, 11988)

In [27]:
adata_combined.write('st_BRCA_combined.h5ad')

In [10]:
adata_combined.X

array([[1., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 4., 1., ..., 0., 0., 0.]], dtype=float32)

In [13]:

# 预处理合并后的数据
sc.pp.normalize_total(adata_combined, target_sum=1e4)
sc.pp.log1p(adata_combined)


In [20]:
adata_combined.X.shape

(11214, 2000)